# Nowcoder Interview Search Data
## Fully Reproducible HAR → Structured Dataset Pipeline V2

**Designed for the MGS 4701 group project**

This notebook is an improved version of the earlier HAR parser. It is designed to work with the **complete 1–20 page HAR export** and can also scale to multiple search keywords later.

### What this notebook does

It does **not** send requests to Nowcoder's search API.  
It only reads `.har` files that you have already exported from the browser's Network panel.

Pipeline:

**Browser search → Export HAR → Python reads HAR → Locate stored search-result JSON → Extract records → Audit page coverage → Merge → Deduplicate → Preliminary BA/DA relevance screening → Export CSV**

### Main improvements in V2

- Works with a full 1–20 page HAR export.
- Automatically finds all `.har` files in the notebook folder or `raw_har/`.
- Handles both `contentData` and `momentData` result structures.
- Preserves search query, page number, rank, record type, IDs, title, content, author, timestamps, and source URL.
- Audits **missing pages**, **duplicate page captures**, and **pages returning fewer records than `page_size`**.
- Does **not** fabricate missing records.
- Deduplicates posts by UUID first, then content ID.
- Can later combine many keyword HAR files into one reproducible dataset.
- Produces raw, unique, relevant-candidate, request-audit, and query-coverage CSVs.

### Recommended folder structure

```text
project_folder/
│
├── nowcoder_HAR_FULL20_REPRODUCIBLE_PIPELINE_V2.ipynb
│
├── raw_har/
│   ├── 商业分析面经_1-20.har
│   ├── 数据分析面经_1-20.har
│   ├── 商业分析师面经_1-20.har
│   └── ...
│
└── output/
```

You may also place `.har` files directly next to this notebook.

In [ ]:
from pathlib import Path
import base64
import html
import json
import re
from datetime import datetime, timezone

import pandas as pd

print("✅ Imports ready")

## 1. Configuration

For the current test, the full HAR should contain pages **1–20**.

Later, when you add more keywords, the same notebook can process all HAR files together.

In [ ]:
CURRENT_DIR = Path(".")
RAW_HAR_DIR = Path("raw_har")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOWCODER_BASE = "https://www.nowcoder.com"

# Marker for the stored search-result request inside the HAR file.
SEARCH_ENDPOINT_MARKER = "gw-c.nowcoder.com/api/sparta/pc/search"

# Expected pages for a full Nowcoder search export.
EXPECTED_PAGES = set(range(1, 21))

print("Working directory:", CURRENT_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

## 2. Helper functions

In [ ]:
def decode_har_content(content_obj):
    """
    Decode response.content stored inside a HAR entry.
    Supports ordinary text and base64-encoded bodies.
    """
    if not isinstance(content_obj, dict):
        return ""

    text = content_obj.get("text", "")
    encoding = content_obj.get("encoding")

    if not text:
        return ""

    if encoding == "base64":
        try:
            raw = base64.b64decode(text)
            return raw.decode("utf-8", errors="replace")
        except Exception:
            return ""

    return text


def ms_to_iso(value):
    """
    Convert Unix time in milliseconds (or seconds) to UTC text.
    """
    if value in (None, ""):
        return ""

    try:
        value = float(value)

        if value > 10_000_000_000:
            value = value / 1000

        dt = datetime.fromtimestamp(
            value,
            tz=timezone.utc,
        )

        return dt.strftime("%Y-%m-%d %H:%M:%S UTC")

    except Exception:
        return ""


def clean_text(value):
    """
    Convert HTML-ish text into plain readable text.
    """
    if value is None:
        return ""

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def choose_content_object(data):
    """
    Select the useful nested content object.

    Observed Nowcoder search-result structures:
      - contentData -> POST
      - momentData  -> MOMENT

    Includes a generic fallback for future variants.
    """
    if not isinstance(data, dict):
        return {}, "OTHER", ""

    known_shapes = [
        ("contentData", "POST"),
        ("momentData", "MOMENT"),
    ]

    for key, label in known_shapes:
        obj = data.get(key)

        if isinstance(obj, dict) and obj:
            return obj, label, key

    for key, value in data.items():
        if not isinstance(value, dict):
            continue

        looks_like_content = any(
            field in value
            for field in (
                "title",
                "newTitle",
                "content",
                "newContent",
                "uuid",
            )
        )

        has_identity = any(
            field in value
            for field in (
                "id",
                "uuid",
            )
        )

        if looks_like_content and has_identity:
            return value, key.upper(), key

    return {}, "OTHER", ""


def build_source_url(uuid_value):
    uuid_value = str(uuid_value or "").strip()

    if not uuid_value:
        return ""

    return (
        f"{NOWCODER_BASE}/feed/main/detail/"
        f"{uuid_value}"
    )

## 3. Find HAR files

The notebook automatically looks in:

1. the current notebook folder
2. `raw_har/`

If you keep the old 5-page test HAR together with the new 20-page HAR, that is okay.  
The record-level deduplication later prevents duplicate posts from inflating the final unique dataset.

In [ ]:
def find_har_files():
    files = []

    files.extend(CURRENT_DIR.glob("*.har"))

    if RAW_HAR_DIR.exists():
        files.extend(RAW_HAR_DIR.glob("*.har"))

    seen = set()
    unique_files = []

    for path in files:
        resolved = str(path.resolve())

        if resolved not in seen:
            seen.add(resolved)
            unique_files.append(path)

    return sorted(
        unique_files,
        key=lambda p: str(p).lower(),
    )


har_files = find_har_files()

print("HAR files found:", len(har_files))

for path in har_files:
    print(" -", path)

if not har_files:
    raise FileNotFoundError(
        "No .har files found. Put the HAR file next to this notebook "
        "or inside a folder named raw_har/."
    )

## 4. Parse one HAR file

The parser keeps only entries matching the stored Nowcoder search-result request:

```text
POST .../api/sparta/pc/search
```

It reads the request payload and the response body **already stored inside the HAR**.

In [ ]:
def parse_one_har(path):
    with path.open(
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as f:
        har = json.load(f)

    extracted_records = []
    request_audit = []

    entries = (
        har.get("log", {})
        .get("entries", [])
    )

    for entry_index, entry in enumerate(entries):
        request = entry.get("request", {}) or {}
        response = entry.get("response", {}) or {}

        method = request.get("method", "")
        request_url = request.get("url", "")

        if method != "POST":
            continue

        if SEARCH_ENDPOINT_MARKER not in request_url:
            continue

        # -------------------------
        # Parse request payload
        # -------------------------
        post_data = request.get("postData", {}) or {}
        request_text = post_data.get("text", "")

        try:
            request_json = (
                json.loads(request_text)
                if request_text
                else {}
            )
        except Exception:
            request_json = {}

        # -------------------------
        # Parse saved response body
        # -------------------------
        response_content = (
            response.get("content", {})
            or {}
        )

        response_text = decode_har_content(
            response_content
        )

        if not response_text:
            continue

        try:
            response_json = json.loads(
                response_text
            )
        except Exception:
            continue

        data = (
            response_json.get("data")
            or {}
        )

        records = (
            data.get("records")
            or []
        )

        search_query = (
            request_json.get("query")
            or ""
        )

        requested_page = (
            request_json.get("page")
        )

        response_page = (
            data.get("current")
        )

        page_size = (
            data.get("size")
        )

        reported_total = (
            data.get("total")
        )

        reported_total_pages = (
            data.get("totalPage")
        )

        # -------------------------
        # Request-level audit row
        # -------------------------
        request_audit.append({
            "source_har": path.name,
            "har_entry_index": entry_index,
            "search_query": search_query,
            "requested_page": requested_page,
            "response_page": response_page,
            "page_size": page_size,
            "reported_total": reported_total,
            "reported_total_pages": reported_total_pages,
            "records_in_response": len(records),
            "short_page_flag": (
                isinstance(page_size, (int, float))
                and len(records) < page_size
            ),
            "http_status": response.get("status"),
            "request_url": request_url,
        })

        # -------------------------
        # Record-level rows
        # -------------------------
        for rank, record in enumerate(
            records,
            start=1,
        ):
            outer_data = (
                record.get("data")
                or {}
            )

            user_brief = (
                outer_data.get("userBrief")
                or {}
            )

            (
                content_obj,
                record_kind,
                source_object,
            ) = choose_content_object(
                outer_data
            )

            content_id = str(
                content_obj.get("id")
                or outer_data.get("contentId")
                or ""
            )

            uuid_value = str(
                content_obj.get("uuid")
                or ""
            )

            title = clean_text(
                content_obj.get("title")
                or content_obj.get("newTitle")
                or record.get("title")
                or ""
            )

            content_text = clean_text(
                content_obj.get("content")
                or content_obj.get("newContent")
                or ""
            )

            author_id = str(
                content_obj.get("authorId")
                or content_obj.get("userId")
                or user_brief.get("userId")
                or ""
            )

            author_nickname = clean_text(
                user_brief.get("nickname")
                or ""
            )

            extracted_records.append({
                "source_har": path.name,
                "search_query": search_query,
                "page": response_page,
                "rank_in_page": rank,
                "record_kind": record_kind,
                "source_object": source_object,
                "rc_type": record.get("rc_type"),
                "content_type": outer_data.get("contentType"),
                "content_id": content_id,
                "uuid": uuid_value,
                "title": title,
                "content": content_text,
                "content_chars": len(content_text),
                "author_id": author_id,
                "author_nickname": author_nickname,
                "create_time_utc": ms_to_iso(
                    content_obj.get("createTime")
                ),
                "show_time_utc": ms_to_iso(
                    content_obj.get("showTime")
                ),
                "source_url": build_source_url(
                    uuid_value
                ),
                "entity_data_id": record.get("entityDataId"),
            })

    return extracted_records, request_audit

## 5. Parse all HAR files

In [ ]:
all_records = []
all_requests = []

for path in har_files:
    records, requests = parse_one_har(path)

    all_records.extend(records)
    all_requests.extend(requests)

    print(
        f"{path.name}: "
        f"{len(requests)} search requests, "
        f"{len(records)} records"
    )

raw_df = pd.DataFrame(all_records)
request_df = pd.DataFrame(all_requests)

print("\n" + "=" * 70)
print("RAW EXTRACTION SUMMARY")
print("=" * 70)

print("HAR files:", len(har_files))
print("Search-result requests:", len(request_df))
print("Raw search records:", len(raw_df))

## 6. Full 1–20 page coverage audit

This is the most important V2 validation step.

For each search query, the notebook checks:

- pages captured
- missing pages from 1–20
- duplicate page captures
- reported total
- reported total pages
- actual number of records stored in the HAR
- pages returning fewer than the nominal `page_size`

**Important:** if the site reports `total = 400` but the HAR contains fewer than 400 records, this notebook reports the discrepancy and does not invent or impute missing rows.

In [ ]:
def sorted_int_values(series):
    values = []

    for value in series.dropna():
        try:
            values.append(int(value))
        except Exception:
            pass

    return sorted(values)


coverage_rows = []

if not request_df.empty:
    for query, group in request_df.groupby(
        "search_query",
        dropna=False,
    ):
        captured_pages_all = sorted_int_values(
            group["response_page"]
        )

        captured_pages_unique = sorted(
            set(captured_pages_all)
        )

        captured_page_set = set(
            captured_pages_unique
        )

        missing_pages = sorted(
            EXPECTED_PAGES
            - captured_page_set
        )

        duplicate_pages = sorted(
            {
                p
                for p in captured_pages_all
                if captured_pages_all.count(p) > 1
            }
        )

        short_pages = sorted(
            set(
                sorted_int_values(
                    group.loc[
                        group[
                            "short_page_flag"
                        ].fillna(False),
                        "response_page",
                    ]
                )
            )
        )

        coverage_rows.append({
            "search_query": query,
            "unique_pages_captured":
                len(captured_pages_unique),
            "captured_pages":
                ",".join(
                    map(
                        str,
                        captured_pages_unique,
                    )
                ),
            "missing_pages_1_to_20":
                ",".join(
                    map(
                        str,
                        missing_pages,
                    )
                ),
            "duplicate_page_captures":
                ",".join(
                    map(
                        str,
                        duplicate_pages,
                    )
                ),
            "short_pages":
                ",".join(
                    map(
                        str,
                        short_pages,
                    )
                ),
            "captured_records_from_requests":
                int(
                    group[
                        "records_in_response"
                    ].sum()
                ),
            "max_reported_total":
                group[
                    "reported_total"
                ].max(),
            "max_reported_total_pages":
                group[
                    "reported_total_pages"
                ].max(),
            "all_expected_pages_captured":
                len(missing_pages) == 0,
        })

coverage_df = pd.DataFrame(
    coverage_rows
)

display(coverage_df)

print("\nRequest-by-request audit:")

audit_columns = [
    "source_har",
    "search_query",
    "requested_page",
    "response_page",
    "page_size",
    "reported_total",
    "reported_total_pages",
    "records_in_response",
    "short_page_flag",
    "http_status",
]

display(
    request_df[
        audit_columns
    ].sort_values(
        [
            "search_query",
            "response_page",
            "source_har",
        ]
    )
)

## 7. Deduplicate posts

The same post can appear:

- in multiple search queries
- in multiple HAR exports
- in an old test HAR and a newer complete HAR

Deduplication priority:

1. UUID
2. content ID
3. fallback: title + author ID

The notebook also stores `search_hit_count`, showing how many raw search hits pointed to the same unique post.

In [ ]:
def make_dedupe_key(row):
    uuid_value = str(
        row.get("uuid", "")
        or ""
    ).strip()

    content_id = str(
        row.get("content_id", "")
        or ""
    ).strip()

    title = str(
        row.get("title", "")
        or ""
    ).strip()

    author_id = str(
        row.get("author_id", "")
        or ""
    ).strip()

    if uuid_value:
        return "uuid:" + uuid_value

    if content_id:
        return "content_id:" + content_id

    return (
        "fallback:"
        + title
        + "|"
        + author_id
    )


if not raw_df.empty:
    raw_df["dedupe_key"] = raw_df.apply(
        make_dedupe_key,
        axis=1,
    )

    hit_counts = (
        raw_df[
            "dedupe_key"
        ]
        .value_counts()
        .rename(
            "search_hit_count"
        )
    )

    unique_df = (
        raw_df
        .drop_duplicates(
            subset=[
                "dedupe_key"
            ],
            keep="first",
        )
        .copy()
    )

    unique_df[
        "search_hit_count"
    ] = unique_df[
        "dedupe_key"
    ].map(
        hit_counts
    )

else:
    unique_df = raw_df.copy()


print("=" * 70)
print("DEDUPLICATION")
print("=" * 70)

print(
    "Raw search records:",
    len(raw_df),
)

print(
    "Unique posts:",
    len(unique_df),
)

print(
    "Duplicate search hits removed:",
    len(raw_df) - len(unique_df),
)

## 8. Transparent preliminary BA / DA / Analytics relevance screen

This is only a **candidate-screening rule**. It is not a substitute for the hand-coded gold set required by the course.

A unique record is marked as a candidate when its title/content contains:

- at least one BA/DA/Analytics role term, and
- at least one interview/recruiting term.

In [ ]:
ROLE_TERMS = [
    # Data Analytics
    "数据分析",
    "数据分析师",
    "数分",
    "data analyst",
    "data analytics",

    # Business Analytics
    "商业分析",
    "商业分析师",
    "商分",
    "business analyst",
    "business analytics",

    # Strategy / Operations Analytics
    "策略分析",
    "策略分析师",
    "战略分析",
    "战略分析师",
    "经营分析",
    "经营分析师",
    "业务分析",
    "业务分析师",

    # Product / Growth Analytics
    "产品分析",
    "用户增长分析",
    "增长分析",
    "增长策略分析",

    # BI / generic analytics
    "商业智能",
    "business intelligence",
    "bi分析",
    "analytics",
]

INTERVIEW_TERMS = [
    "面经",
    "面试",
    "一面",
    "二面",
    "三面",
    "四面",
    "hr面",
    "hr 面",
    "群面",
    "电面",
    "电话面试",
    "笔试",
    "秋招",
    "春招",
    "校招",
    "暑期实习",
    "实习面试",
]


def find_hits(text, terms):
    text = str(
        text or ""
    ).lower()

    return [
        term
        for term in terms
        if term.lower() in text
    ]


def relevance_screen(row):
    combined_text = (
        str(
            row.get(
                "title",
                "",
            )
        )
        + "\n"
        + str(
            row.get(
                "content",
                "",
            )
        )
    )

    role_hits = find_hits(
        combined_text,
        ROLE_TERMS,
    )

    interview_hits = find_hits(
        combined_text,
        INTERVIEW_TERMS,
    )

    return pd.Series({
        "role_hits":
            "|".join(
                role_hits[:15]
            ),

        "interview_hits":
            "|".join(
                interview_hits[:15]
            ),

        "is_relevant_candidate":
            bool(role_hits)
            and bool(interview_hits),
    })


if not unique_df.empty:
    relevance_columns = unique_df.apply(
        relevance_screen,
        axis=1,
    )

    unique_df = pd.concat(
        [
            unique_df.reset_index(
                drop=True
            ),
            relevance_columns.reset_index(
                drop=True
            ),
        ],
        axis=1,
    )

    relevant_df = unique_df[
        unique_df[
            "is_relevant_candidate"
        ]
        .fillna(False)
        .astype(bool)
    ].copy()

else:
    relevant_df = unique_df.copy()


print("=" * 70)
print("PRELIMINARY RELEVANCE SCREEN")
print("=" * 70)

print(
    "Unique posts:",
    len(unique_df),
)

print(
    "BA/DA/Analytics interview candidates:",
    len(relevant_df),
)

## 9. Quality checks

In [ ]:
def missing_count(df, column):
    if (
        df.empty
        or column not in df.columns
    ):
        return 0

    values = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    return int(
        values.eq("").sum()
    )


print("=" * 70)
print("QUALITY CHECKS")
print("=" * 70)

print(
    "Missing title:",
    missing_count(
        unique_df,
        "title",
    ),
)

print(
    "Missing content:",
    missing_count(
        unique_df,
        "content",
    ),
)

print(
    "Missing UUID:",
    missing_count(
        unique_df,
        "uuid",
    ),
)

print(
    "Missing source URL:",
    missing_count(
        unique_df,
        "source_url",
    ),
)


if not unique_df.empty:
    print("\nRecord type distribution:")
    print(
        unique_df[
            "record_kind"
        ]
        .value_counts(
            dropna=False
        )
    )

    print("\nUnique posts by original search query:")
    print(
        unique_df[
            "search_query"
        ]
        .value_counts(
            dropna=False
        )
    )

## 10. Export all reproducible outputs

Files created:

```text
output/
├── nowcoder_search_records_RAW.csv
├── nowcoder_search_records_UNIQUE.csv
├── nowcoder_search_records_RELEVANT_CANDIDATES.csv
├── nowcoder_search_REQUEST_AUDIT.csv
└── nowcoder_search_QUERY_COVERAGE.csv
```

In [ ]:
raw_output = (
    OUTPUT_DIR
    / "nowcoder_search_records_RAW.csv"
)

unique_output = (
    OUTPUT_DIR
    / "nowcoder_search_records_UNIQUE.csv"
)

relevant_output = (
    OUTPUT_DIR
    / "nowcoder_search_records_RELEVANT_CANDIDATES.csv"
)

request_output = (
    OUTPUT_DIR
    / "nowcoder_search_REQUEST_AUDIT.csv"
)

coverage_output = (
    OUTPUT_DIR
    / "nowcoder_search_QUERY_COVERAGE.csv"
)


raw_df.to_csv(
    raw_output,
    index=False,
    encoding="utf-8-sig",
)

unique_df.to_csv(
    unique_output,
    index=False,
    encoding="utf-8-sig",
)

relevant_df.to_csv(
    relevant_output,
    index=False,
    encoding="utf-8-sig",
)

request_df.to_csv(
    request_output,
    index=False,
    encoding="utf-8-sig",
)

coverage_df.to_csv(
    coverage_output,
    index=False,
    encoding="utf-8-sig",
)


print("✅ Export complete")

for path in [
    raw_output,
    unique_output,
    relevant_output,
    request_output,
    coverage_output,
]:
    print(" -", path.resolve())

## 11. Final class-presentation summary

This cell prints the core reproducibility statistics.

In [ ]:
print("=" * 72)
print("NOWCODER HAR DATA PIPELINE — FINAL SUMMARY")
print("=" * 72)

print(
    "HAR files processed:",
    len(har_files),
)

print(
    "Stored search requests parsed:",
    len(request_df),
)

print(
    "Raw search records extracted:",
    len(raw_df),
)

print(
    "Unique posts after deduplication:",
    len(unique_df),
)

print(
    "Preliminary BA/DA/Analytics interview candidates:",
    len(relevant_df),
)

print()

if not coverage_df.empty:
    for _, row in coverage_df.iterrows():
        print(
            f"Query: {row['search_query']}"
        )

        print(
            "  Unique pages captured:",
            row[
                "unique_pages_captured"
            ],
        )

        print(
            "  Missing pages 1–20:",
            row[
                "missing_pages_1_to_20"
            ]
            or "None",
        )

        print(
            "  Short pages:",
            row[
                "short_pages"
            ]
            or "None",
        )

        print(
            "  Records stored in captured responses:",
            row[
                "captured_records_from_requests"
            ],
        )

        print(
            "  Search interface reported total:",
            row[
                "max_reported_total"
            ],
        )

        reported = row[
            "max_reported_total"
        ]

        captured = row[
            "captured_records_from_requests"
        ]

        if (
            pd.notna(reported)
            and pd.notna(captured)
            and captured < reported
        ):
            print(
                "  ⚠️ Difference:",
                int(reported - captured),
                "records. "
                "The notebook does NOT fabricate missing rows."
            )

        print()

print(
    "Reminder: automatic relevance screening is only a preliminary filter."
)

## 12. Preview relevant candidate records

In [ ]:
preview_columns = [
    "search_query",
    "page",
    "rank_in_page",
    "record_kind",
    "title",
    "content_chars",
    "role_hits",
    "interview_hits",
    "source_url",
]

preview_columns = [
    col
    for col in preview_columns
    if col in relevant_df.columns
]

if not relevant_df.empty:
    display(
        relevant_df[
            preview_columns
        ].head(30)
    )
else:
    print(
        "No relevant candidate records found."
    )

# Methodology wording for your report / presentation

> We navigated public Nowcoder search-result pages in a normal browser session and exported the browser Network log as HAR files. The Python notebook then parsed the search-result JSON responses already stored in those HAR files, preserved query and page provenance, extracted record-level fields, audited page coverage and response sizes, merged results across HAR files, and removed duplicate posts using UUID or content ID. A transparent keyword rule was used only as a preliminary relevance screen; final validation should be based on the manually coded gold set.

Keep the **original HAR files**, this **notebook**, and the **output CSV files** together so the workflow can be reproduced.